In [0]:
BASE = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder"

CATALOG = "workspace"
BRONZE = f"{CATALOG}.mimic_bronze"

from pyspark.sql import functions as F
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")

In [0]:
def to_lowercase_cols(df):
    # fixes the ORIGINALAMOUNT / ORIGINALRATE quirk in procedureevents
    for c in df.columns:
        if c != c.lower():
            df = df.withColumnRenamed(c, c.lower())
    return df

def ingest(path, table, infer=True):
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", infer)
          .load(path))
    df = to_lowercase_cols(df)
    # lineage metadata so every row is traceable back to its source file
    df = (df.withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path")))
    df.write.mode("overwrite").format("delta").saveAsTable(f"{BRONZE}.{table}")
    print(f"{BRONZE}.{table}: {spark.table(f'{BRONZE}.{table}').count():,} rows")

# ICD tables: infer=False so icd_code stays a string and keeps its leading zeros
ingest(f"{BASE}/hosp/procedures_icd.csv*",   "procedures_icd",   infer=False)
ingest(f"{BASE}/hosp/d_icd_procedures.csv*", "d_icd_procedures", infer=False)

# event tables: inference is fine here, the ids are genuinely numeric
ingest(f"{BASE}/icu/procedureevents.csv*",  "procedureevents")
ingest(f"{BASE}/icu/inputevents.csv*",      "inputevents")
ingest(f"{BASE}/icu/ingredientevents.csv*", "ingredientevents")
ingest(f"{BASE}/icu/outputevents.csv*",     "outputevents")

In [0]:
# check: ICD codes starting with 0 are still intact (not converted to numbers)
spark.table(f"{BRONZE}.procedures_icd") \
     .filter("icd_code LIKE '0%'") \
     .select("icd_code", "icd_version").show(5)

# check: all column names are lowercase now (result should be empty [])
print([c for c in spark.table(f"{BRONZE}.procedureevents").columns if c != c.lower()])